In [1]:
import os
os.chdir("../")

In [2]:
%pwd

'/Users/prasad/Learning/Projects/Text_Summarization_Project'

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    data_path: Path
    model_path: Path
    tokenizer_path: Path
    metric_file: Path

In [4]:
from textSummarization.constants import *
from textSummarization.utils.common import read_yaml, create_directories


class ConfigurationManager:
    def __init__(
        self,
        config_file_path: Path = CONFIG_FILE_PATH,
        params_file_path: Path = PARAMS_FILE_PATH
    ) -> None:
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
    
        create_directories([self.config.artificats_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        evaluation_config = self.config.model_evaluation

        create_directories([evaluation_config.root_dir])
        
        model_evaluation_config = ModelEvaluationConfig(
            root_dir=Path(evaluation_config.root_dir),
            data_path=Path(evaluation_config.data_path),
            model_path=Path(evaluation_config.model_path),
            tokenizer_path=Path(evaluation_config.tokenizer_path),
            metric_file=Path(evaluation_config.metric_file)
        )

        return model_evaluation_config

In [7]:
!pip install evaluate

In [8]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_from_disk
import torch
import pandas as pd
from tqdm import tqdm
import evaluate

In [11]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config
    
    def generate_batch_sized_chunks(list_of_elements, batch_size):
        """split the dataset into smaller batches that we can process simultaneously
        Yield successive batch-sized chunks from list_of_elements."""
        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i : i + batch_size]


    def calculate_metric_on_test_ds(dataset, metric, model, tokenizer,
                                    batch_size=16, device="cpu",
                                    column_text="article",
                                    column_summary="highlights"):
        article_batches = list(self.generate_batch_sized_chunks(dataset[column_text], batch_size))
        target_batches = list(self.generate_batch_sized_chunks(dataset[column_summary], batch_size))

        for article_batch, target_batch in tqdm(
            zip(article_batches, target_batches), total=len(article_batches)):

            inputs = tokenizer(article_batch, max_length=1024, truncation=True,
                                padding="max_length", return_tensors='pt')

            summaries = model.generate(input_ids=inputs['input_ids'].to(device),
                                        attention_mask=inputs['attention_mask'].to(device),
                                        length_penalty=0.8, num_beams=8, max_length=128)

            decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True,
                                                    clean_up_tokenization_spaces=True)
                                for s in summaries]

            decoded_summaries = [d.replace("", " ") for d in decoded_summaries]

            metric.add_batch(predictions=decoded_summaries, references=target_batch)

        score = metric.compute()
        return score
    
    def evaluate(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_path).to(device)
        tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_path)

        dataset_samsum_pt = load_from_disk(self.config.data_path)

        rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
        metric = evaluate.load("rouge")

        score = self.calculate_metric_on_test_ds(dataset_samsum_pt['test'][0:10], 
                                            metric, 
                                            model, 
                                            tokenizer,
                                            batch_size=2,
                                            device=device,
                                            column_text="dialogue", 
                                            column_summary="summary")
        
        rouge_dict = dict((rn, score[rn]) for rn in rouge_names)

        df = pd.DataFrame(rouge_dict, index=['pegasus'])
        df.to_csv(self.config.metric_file, index=False)

In [ ]:
try:
    config_manager = ConfigurationManager()
    model_evaluation_config = config_manager.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.evaluate()
except Exception as e:
    raise e